# 04 — Phase 4 analysis

CPU-only — run on Colab's CPU runtime or locally on the Mac, either is fine. Produces the `ΔR_t`/`ΔAUC_t` table, `T_t` table, reading-rule verdict, variance-decomposition statement, descriptive Spearman correlation, and the Tommy-shape figure (plan §3 Phase 4).

In [ ]:
%pip install -q pandas numpy scipy matplotlib

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP2_DIR = '/content/RLVR/experiment 2'  # or a local path if running on the Mac
CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
RUN_DIR = Path(f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_REPLACE_WITH_HASH')
sa, sb = CONFIG['stage_a'], CONFIG['stage_b']

## P1/P2 — ΔR_t and ΔAUC_t

In [ ]:
rows = []
for step in sa['adapt_from_checkpoints']:
    per_seed = []
    for seed in sb['committed_seeds']:
        cell_dir = RUN_DIR / 'stage_b' / f'ckpt{step}_seed{seed}'
        summary = json.loads((cell_dir / 'summary.json').read_text())
        curve = [json.loads(l) for l in (cell_dir / 'stageb_eval_curve.jsonl').read_text().splitlines()]
        r_end = curve[-1]['accuracy']
        auc = float(np.mean([summary['acc_before']] + [c['accuracy'] for c in curve]))
        per_seed.append({'seed': seed, 'r_end': r_end, 'auc': auc})
    rows.append({'ckpt': step,
                 'r_end_mean': float(np.mean([p['r_end'] for p in per_seed])),
                 'r_end_sd': float(np.std([p['r_end'] for p in per_seed])) if len(per_seed) > 1 else None,
                 'auc_mean': float(np.mean([p['auc'] for p in per_seed])),
                 'n_seeds': len(per_seed)})
df = pd.DataFrame(rows).set_index('ckpt')
r_end_0, auc_0 = df.loc[0, 'r_end_mean'], df.loc[0, 'auc_mean']
df['delta_R'] = r_end_0 - df['r_end_mean']
df['delta_AUC'] = auc_0 - df['auc_mean']
if (df['n_seeds'] <= 1).all():
    print('NOTE: n_seeds=1 at committed scope — no within-checkpoint (seed) '
          'variance estimate exists. Do not report an SD or a variance '
          'decomposition from this table; that requires the stretch-goal seeds.')
print(df)

## P3 — T_t and the reading rule

In [ ]:
transfer = json.loads((RUN_DIR / 'analysis' / 'transfer_T.json').read_text())
print('T_t:', transfer['T_t'])

def reading_rule(t_t, near_zero=0.03):
    if abs(t_t) <= near_zero:
        return 'T_t ~ 0: delta_R reads as candidate plasticity loss'
    if t_t > near_zero:
        return 'T_t >> 0: CONFOUNDED — stage 1 raised the stage-B starting point'
    return 'T_t << 0: stage 1 damaged stage-B ability directly — separate level from learning effect'

for step in sa['adapt_from_checkpoints']:
    t_t = transfer['T_t'][str(step)]
    print(f"ckpt {step}: T_t={t_t:.4f} -> {reading_rule(t_t)}")

## Descriptive Spearman + Tommy-shape figure

In [ ]:
q_rows = []
for step in sa['checkpoint_steps']:
    q = json.loads((RUN_DIR / 'measurements' / f'metrics_ckpt{step}.json').read_text())
    layer = CONFIG['measurement']['layers'][1]  # the 'mid-depth' probe layer
    q_rows.append({'ckpt': step, 'erank': q['per_layer'][f'layer{layer}']['erank']})
q_df = pd.DataFrame(q_rows).set_index('ckpt')
joined = df.join(q_df, how='inner')
n = len(joined)
if n < 4:
    print(f'n={n} checkpoints — Spearman correlation is descriptive ONLY, do not quote a p-value as inferential.')
rho, p = spearmanr(joined['erank'], joined['delta_R'])
print(f'Spearman rho={rho:.3f} (p={p:.3f}, n={n}) — DESCRIPTIVE, small n')

fig, ax = plt.subplots(figsize=(6, 4.5))
for step in sa['adapt_from_checkpoints']:
    for seed in sb['committed_seeds']:
        cell_dir = RUN_DIR / 'stage_b' / f'ckpt{step}_seed{seed}'
        summary = json.loads((cell_dir / 'summary.json').read_text())
        curve = [json.loads(l) for l in (cell_dir / 'stageb_eval_curve.jsonl').read_text().splitlines()]
        xs = [0] + [c['step'] for c in curve]
        ys = [summary['acc_before']] + [c['accuracy'] for c in curve]
        style = '--' if step == 0 else '-'
        ax.plot(xs, ys, style, marker='o', label=f'ckpt {step}' + (' (baseline)' if step == 0 else ''))
ax.set(xlabel='stage-2 GRPO updates', ylabel='Simulation accuracy',
       title='exp2 (Colab, 7B LoRA): stage-B reward vs stage-1 checkpoint')
ax.legend(frameon=False)
(RUN_DIR / 'analysis').mkdir(exist_ok=True)
fig.savefig(RUN_DIR / 'analysis' / 'fig_tommy_shape.png', dpi=150, bbox_inches='tight')
print('saved', RUN_DIR / 'analysis' / 'fig_tommy_shape.png')

In [ ]:
summary_out = {
    'run_dir': str(RUN_DIR), 'n_checkpoints': len(joined), 'n_seeds': int(df['n_seeds'].iloc[0]),
    'delta_R_by_checkpoint': df['delta_R'].to_dict(),
    'delta_AUC_by_checkpoint': df['delta_AUC'].to_dict(),
    'T_t_by_checkpoint': transfer['T_t'],
    'spearman_rho': float(rho), 'spearman_p': float(p),
    'spearman_is_descriptive_only': True,
    'variance_decomposition_available': bool((df['n_seeds'] > 1).all()),
}
(RUN_DIR / 'analysis' / 'analysis_summary.json').write_text(json.dumps(summary_out, indent=1, default=str))
print(summary_out)

## Commit reminder

Commit `analysis/analysis_summary.json` and `analysis/fig_tommy_shape.png`, prefix `exp2-colab:`. Report `ΔR_t` next to `T_t` always (plan §7). Never claim RLVR 'reduces the model's ability to learn' — the claim is fixed-budget adaptability vs. the stated baseline (plan §5 framing constraint).